# Suspicious Sensor Analysis
## Probability, Statistics, and Monte Carlo Infrastructure Study

---

**Notebook**: `01_descriptive_statistics.ipynb`
**Author**: Brice Nelson
**Date**: 2026-05-24
**Environment**
- Python 3.12+
- UV Environment

---

# Purpose

This notebook explores pressure sensor data from a synthetic infrastructure monitoring system.

The analysis focuses on:
- descriptive statistics,
- probability distributions,
- z-score analysis,
- anomaly detection,
- Monte Carlo simulation,
- and engineering interpretation of operational uncertainty.

The objective is to develop practical understanding of probabilistic analysis techniques within an infrastructure engineering context.

---

# Dataset

```text
data/raw/pressure_sensor_readings.csv
```

---

# Related Documentation

```text
docs/00_overview/01_problem_statement.md
```

---

# Topics Covered

- Mean
- Variance
- Standard deviation
- Probability distributions
- Z-scores
- Monte Carlo simulation
- Bayesian reasoning
- Infrastructure risk interpretation

---

# Primary Libraries

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
```

---

# Notes

This notebook is educational and analysis-focused.

Some methods prioritize readability and conceptual clarity over computational optimization.

Because understanding why a model behaves strangely is often more valuable than generating incorrect results with extreme confidence. Modern analytics occasionally forgets this.

In [12]:

from src.suspicious_sensor.io.loader import load_data
from src.suspicious_sensor.diagnostics.diagnostics import dataset_diagnostics
from src.suspicious_sensor.preprocessing.data_cleaning import missing_data_investigation
from pprint import pprint


In [3]:
df = load_data("pressure_sensor_readings.csv")
print(type(df))

<class 'pandas.DataFrame'>


In [4]:
df.head(5)

,timestamp,pressure_psi,flow_gpm,sensor_status,pump_status,anomaly_flag
0,2026-01-01 00:00:00,73.20,1379.8,NORMAL,OFF,0
1,2026-01-01 01:00:00,73.26,1350.2,NORMAL,OFF,0
2,2026-01-01 02:00:00,74.45,1422.4,NORMAL,OFF,0
3,2026-01-01 03:00:00,71.18,1514.6,NORMAL,ON,0
4,2026-01-01 04:00:00,71.86,1461.0,NORMAL,OFF,0


In [10]:
shape = df.shape
print(type(shape))
print(f"Dataset Shape: {shape}")

<class 'tuple'>
Dataset Shape: (4340, 6)


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      4340 non-null   str    
 1   pressure_psi   4265 non-null   float64
 2   flow_gpm       4340 non-null   float64
 3   sensor_status  4340 non-null   str    
 4   pump_status    4340 non-null   str    
 5   anomaly_flag   4340 non-null   int64  
dtypes: float64(2), int64(1), str(3)
memory usage: 203.6 KB


In [7]:
pprint(f'Dataset Diagnostics: {dataset_diagnostics(df)}')

("Dataset Diagnostics: {'shape': (4340, 6), 'data_types': timestamp            "
 'str\n'
 'pressure_psi     float64\n'
 'flow_gpm         float64\n'
 'sensor_status        str\n'
 'pump_status          str\n'
 'anomaly_flag       int64\n'
 "dtype: object, 'memory_usage': np.float64(0.8208446502685547), "
 "'unique_values': {'timestamp': 4320, 'pressure_psi': 959, 'flow_gpm': 3244, "
 "'sensor_status': 4, 'pump_status': 2, 'anomaly_flag': 2}}")


In [9]:
# Data types and missing values
print("Data Types: ")
data_types = df.dtypes
print(data_types)
missing_values = df.isnull().sum()
print("-" * 40)
print("Missing Values: ")
print(missing_values)

Data Types: 
timestamp            str
pressure_psi     float64
flow_gpm         float64
sensor_status        str
pump_status          str
anomaly_flag       int64
dtype: object
----------------------------------------
Missing Values: 
timestamp         0
pressure_psi     75
flow_gpm          0
sensor_status     0
pump_status       0
anomaly_flag      0
dtype: int64


In [11]:
# Investigate the missing values
missing_values_percentage = (missing_values / len(df)) * 100
print("-" * 40)
print("Missing Values Percentage: ")
print(missing_values_percentage)

----------------------------------------
Missing Values Percentage: 
timestamp        0.000000
pressure_psi     1.728111
flow_gpm         0.000000
sensor_status    0.000000
pump_status      0.000000
anomaly_flag     0.000000
dtype: float64


In [14]:
"""
Questions to Investigate Before Cleaning
----------------------------------------
1. Do missing pressure readings correspond to particular `sensor_status` values, such as `OFFLINE`?
2. Are missing readings scattered across the monitoring period or clustered in time?
3. Are missing readings associated with particular `pump_status` values?
4. Could excluding these periods distort conclusions about pressure behavior or operational risk?
"""
print("Investigating Missing Data Patterns...")
missing_data_investigation(df)

Investigating Missing Data Patterns...


{'missing_summary':               missing_count  missing_percentage
 pressure_psi             75            1.728111,
 'missing_pressure_count': 75,
 'missing_pressure_percentage': 1.7281105990783412,
 'missing_pressure_by_sensor_status':                missing_pressure_count  missing_pressure_percentage
 sensor_status                                                     
 OFFLINE                            75                        100.0,
 'pressure_missing_rate_by_sensor_status':                missing_pressure_count  total_records  missing_pressure_rate
 sensor_status                                                              
 OFFLINE                            75             75                  100.0
 DRIFT                               0            587                    0.0
 FROZEN                              0             47                    0.0
 NORMAL                              0           3631                    0.0,
 'missing_pressure_by_pump_status':              mis